<a href="https://colab.research.google.com/github/Areeba-Kh571/flyrank-ml-internship-areeba/blob/main/w03_data_contract.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Areeba-Kh571/flyrank-ml-internship-areeba/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**One row = one content item** (`client_hash_id` + `content_hash_id` pair, matching how
`fact_content_daily_performance` is keyed), summarized twice over two non-overlapping windows built
from that one table:

- **Feature window** — Q1 2026, `2026-01-01` to `2026-03-31` (90 days). Every number that feeds a
  feature is computed only from rows inside this window.
- **Label window** — the 30 days right after: `2026-04-01` to `2026-04-30`. This is where the
  outcome I'm trying to predict is measured.

The label window starts the day after the feature window ends, so every feature is knowable
strictly *before* the moment the label describes — a genuine past → future split, not two
historical windows compared to each other (that was the starter CSV's `trend_direction`
shortcut, and the lane guide calls it out as the weaker version of this target).

I picked Q1 2026 because it's solidly mid-panel — nowhere near June 2026, which the data guide
treats as a sealed test month (it's also the only month in `fact_content_daily_performance_sample`,
which I never touch here).

In [ ]:
%pip -q install duckdb

import os, getpass
import duckdb

# Token order: env var -> Colab Secret -> prompt (last resort, avoid it -- see notebook 03's note
# on a getpass prompt hanging forever if Colab reconnects mid-run).
HF_TOKEN = os.environ.get('HF_TOKEN')
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
HF_TOKEN = HF_TOKEN or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
}
FACT = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"

# Metadata-only checks -- near-free even though the daily table is ~79M rows.
for name, src in TABLES.items():
    n = con.sql(f'SELECT COUNT(*) FROM {src}').fetchone()[0]
    print(f'{name:12} {n:>10,} rows')

span = con.sql(f"SELECT COUNT(*) AS n, MIN(report_date) AS min_date, MAX(report_date) AS max_date FROM {FACT}").df()
print(span)
print('expect ~78,835,655 rows spanning 2025-01-27 to 2026-06-30 -- matches the data guide before I touch anything else.')


Paste your Hugging Face READ token (hf_...): ··········
dim_clients         104 rows
dim_content     519,606 rows


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

          n   min_date   max_date
0  78835655 2025-01-27 2026-06-30
expect ~78,835,655 rows spanning 2025-01-27 to 2026-06-30 -- matches the data guide before I touch anything else.


## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Features** (all built only from `report_date BETWEEN 2026-01-01 AND 2026-03-31` — every one is
knowable before the 2026-04-01 decision moment):

1. `imp_90d` — total GSC impressions across the feature window. Knowable because every
   contributing row's `report_date` is in the past relative to April.
2. `pos_90d` — average GSC position across the feature window, **excluding rows where
   `gsc_avg_position = 0`** (checked below — the starter CSV treats 0 as "no position data," and
   the same convention shows up in the daily table). Knowable, same reasoning.
3. `trend_ratio_90d` — last-30-days-of-the-window impressions vs. the 30-day-equivalent rate of
   the 60 days before that, both *inside* the Q1 window. Still entirely backward-looking; it just
   captures momentum without reaching into April.
4. `days_with_impressions_90d` — count of distinct days in the window with at least one
   impression. A consistency signal, backward-looking.
5. `ai_referral_share_90d` — `sessions_ai` (AI-tool-referred sessions, GA4-sourced, counted only
   on rows where `ga4_data_available IS TRUE`) divided by GSC clicks across the window. Backward-
   looking; this is the warehouse's real column for the lane guide's "AI Referral Opportunity"
   direction.

**Label / proxy:** `is_declining_next30` = 1 when the **label window's** total impressions come in
under 80% of the feature window's own last-30-day rate (`imp_last30`, itself just a feature), else
0. This mirrors notebook 03's 20%-decline rule, but the outcome side now genuinely lives in
April — never in the same window as the features that predict it.

**Context** (grouping / joining only, never features): `client_hash_id`, `content_hash_id` — the
grain of every query below. `url_hash_id` / `keyword_hash_id` on `dim_content` would join in the
same way if I needed them for a case lookup; I don't use them here.

**Excluded**, each with a one-line why:

- Any `gsc_impressions` / `gsc_clicks` / `gsc_avg_position` row with `report_date >= 2026-04-01`
  — that's the outcome window; letting it into a feature is exactly the leak I demonstrate in
  section 4.
- `fact_content_query_90d`'s `*_90d` / `*_last30` columns — that table's fixed 90-day window is
  anchored to the snapshot's own end (near June 2026), not to my March 31 cutoff, so its "recent"
  columns sit inside or past my label window. Using them here would be the same leak wearing a
  different table's name (this is the repo's own documented leakage watch on that table).
- `fact_content_daily_performance_sample` — this IS June 2026, the sealed test month. Not opened
  anywhere in this notebook.
- Anything on `dim_content` beyond the hash join keys — I haven't run a `DESCRIBE` on it yet, so I
  don't know what else it ships. "Didn't check" isn't the same as "safe," so it stays out until I
  do.

In [ ]:
FEATURE_START = "DATE '2026-01-01'"
FEATURE_END   = "DATE '2026-03-31'"
LABEL_START   = "DATE '2026-04-01'"
LABEL_END     = "DATE '2026-04-30'"

# Check the position=0 convention before trusting pos_90d (see the gotcha in the data dictionary).
zero_pos_check = con.sql(f"""
    SELECT COUNT(*) AS rows_with_impressions,
           SUM(CASE WHEN gsc_avg_position = 0 THEN 1 ELSE 0 END) AS rows_position_zero
    FROM {FACT}
    WHERE report_date BETWEEN {FEATURE_START} AND {FEATURE_END}
      AND gsc_impressions > 0
""").df()
print(zero_pos_check)
print('if rows_position_zero > 0: the 0-as-no-data convention carries over -- pos_90d already guards for it below.\n')

frame = con.sql(f"""
    WITH eligible_clients AS (
        -- only clients whose GSC history actually covers the full feature window --
        -- otherwise "no rows" could mean "no tracking yet," not "no activity."
        SELECT client_hash_id
        FROM {TABLES['dim_clients']}
        WHERE gsc_data_start <= {FEATURE_START}
    ),
    feat AS (
        SELECT f.client_hash_id, f.content_hash_id,
               SUM(f.gsc_impressions)                                                                AS imp_90d,
               AVG(CASE WHEN f.gsc_avg_position > 0 THEN f.gsc_avg_position END)                      AS pos_90d,
               SUM(CASE WHEN f.report_date >  {FEATURE_END} - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_last30,
               SUM(CASE WHEN f.report_date <= {FEATURE_END} - INTERVAL 30 DAY THEN f.gsc_impressions ELSE 0 END) AS imp_first60,
               COUNT(DISTINCT CASE WHEN f.gsc_impressions > 0 THEN f.report_date END)                 AS days_with_impressions_90d,
               SUM(CASE WHEN f.ga4_data_available IS TRUE THEN f.sessions_ai ELSE 0 END)              AS ai_sessions_90d,
               SUM(f.gsc_clicks)                                                                      AS clicks_90d
        FROM {FACT} f
        JOIN eligible_clients c USING (client_hash_id)
        WHERE f.report_date BETWEEN {FEATURE_START} AND {FEATURE_END}
        GROUP BY 1, 2
        HAVING imp_90d >= 100   -- drop near-empty pages, same spirit as the lane's "measurable opportunity" idea
    ),
    label AS (
        SELECT client_hash_id, content_hash_id, SUM(gsc_impressions) AS imp_label30
        FROM {FACT}
        WHERE report_date BETWEEN {LABEL_START} AND {LABEL_END}
        GROUP BY 1, 2
    )
    SELECT f.*,
           COALESCE(l.imp_label30, 0)                                             AS imp_label30,
           f.imp_last30 / NULLIF(f.imp_first60 / 2.0, 0)                          AS trend_ratio_90d,
           f.ai_sessions_90d / NULLIF(f.clicks_90d, 0)                            AS ai_referral_share_90d,
           CASE WHEN COALESCE(l.imp_label30, 0) < 0.8 * f.imp_last30 THEN 1 ELSE 0 END AS is_declining_next30
    FROM feat f
    LEFT JOIN label l USING (client_hash_id, content_hash_id)
""").df()

print(f'{len(frame):,} content items with a full Q1 window, an eligible client, and imp_90d >= 100')
frame[['imp_90d', 'pos_90d', 'trend_ratio_90d', 'days_with_impressions_90d', 'ai_referral_share_90d', 'is_declining_next30']].describe()


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   rows_with_impressions  rows_position_zero
0                8629987            529112.0
if rows_position_zero > 0: the 0-as-no-data convention carries over -- pos_90d already guards for it below.



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

102,769 content items with a full Q1 window, an eligible client, and imp_90d >= 100


,imp_90d,pos_90d,trend_ratio_90d,days_with_impressions_90d,ai_referral_share_90d,is_declining_next30
count,102769.000000,102769.000000,93638.000000,102769.000000,69201.000000,102769.000000
mean,5411.219823,15.208598,4.852451,69.866789,0.022801,0.494011
std,14903.091892,14.500030,56.169609,24.289493,0.244666,0.499967
min,100.000000,0.101639,0.000000,1.000000,0.000000,0.000000
25%,366.000000,5.545363,0.844901,52.000000,0.000000,0.000000
50%,1194.000000,9.723837,1.304413,84.000000,0.000000,0.000000
75%,4512.000000,19.586734,2.208678,90.000000,0.000000,1.000000
max,830289.000000,110.624270,8684.000000,90.000000,23.000000,1.000000


## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

1. **Grain** — `(report_date, client_hash_id, content_hash_id)` should have no duplicates.
   Checked on one cheap partition (`month=2026-03`) rather than the full 79M-row table.
2. **Counts + span** — my Q1→April slice's own row count, client/content counts, and date range,
   checked against what the docs promise (104 clients max, dates inside my chosen window).
3. **Availability** — `ga4_data_available` is three-valued (`TRUE` / `FALSE` / `NULL`), so I check
   all three with `IS TRUE` / `IS FALSE` / `IS NULL` rather than trusting a plain boolean test.

In [ ]:
# 1. Grain probe -- single partition, cheap.
grain_check = con.sql(f"""
    SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS c
    FROM read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')
    GROUP BY 1, 2, 3
    HAVING c > 1
    LIMIT 5
""").df()
print('duplicate (date, client, content) rows found in March partition:', len(grain_check))
assert len(grain_check) == 0, 'grain claim failed -- investigate before trusting any aggregate'

# 2. Counts + span for the actual slice this contract uses.
docs_check = con.sql(f"""
    SELECT COUNT(*) AS n_rows,
           COUNT(DISTINCT client_hash_id) AS n_clients,
           COUNT(DISTINCT content_hash_id) AS n_content,
           MIN(report_date) AS min_date,
           MAX(report_date) AS max_date
    FROM {FACT}
    WHERE report_date BETWEEN {FEATURE_START} AND {LABEL_END}
""").df()
print(docs_check)
print(f'expect min_date >= 2026-01-01, max_date <= 2026-04-30, n_clients <= 104 (dim_clients total)\n')

# 3. Availability, three-valued.
avail = con.sql(f"""
    SELECT
        AVG(CASE WHEN ga4_data_available IS TRUE  THEN 1.0 ELSE 0 END) AS pct_true,
        AVG(CASE WHEN ga4_data_available IS FALSE THEN 1.0 ELSE 0 END) AS pct_false,
        AVG(CASE WHEN ga4_data_available IS NULL  THEN 1.0 ELSE 0 END) AS pct_null
    FROM {FACT}
    WHERE report_date BETWEEN {FEATURE_START} AND {FEATURE_END}
""").df()
print(avail)
print('pct_true + pct_false + pct_null should sum to 1.0 -- and pct_null > 0 is exactly why ai_referral_share_90d filters on IS TRUE above, not a plain boolean check.')


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

duplicate (date, client, content) rows found in March partition: 0


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

     n_rows  n_clients  n_content   min_date   max_date
0  35512033         65     380147 2026-01-01 2026-04-30
expect min_date >= 2026-01-01, max_date <= 2026-04-30, n_clients <= 104 (dim_clients total)



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

   pct_true  pct_false  pct_null
0  0.026909   0.470216  0.502875
pct_true + pct_false + pct_null should sum to 1.0 -- and pct_null > 0 is exactly why ai_referral_share_90d filters on IS TRUE above, not a plain boolean check.


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

- **Unbalanced panel.** Only clients whose `gsc_data_start` is on or before 2026-01-01 even
  qualify for a full Q1 feature window (counted below) — the rest weren't dropped for having no
  activity, their tracking simply hadn't started. Anything this contract's data produces is scoped
  to established clients, not the whole roster.
- **`ga4_data_available` is three-valued**, not a clean boolean — `ai_referral_share_90d` and any
  future GA4-sourced feature has to filter on `IS TRUE`, or the `NULL` rows land on the wrong side
  silently.
- **`fact_content_query_90d` doesn't line up with this cutoff** — its fixed window is anchored to
  the snapshot's end, not to March 31, so it's excluded here rather than re-derived per cutoff.
- **June 2026 is deliberately untouched** — this contract's numbers describe Q1 → April only, and
  the sealed test month stays sealed.
- **Everything is pseudonymous.** Nothing here ties back to a real client, page, or query, and
  nothing in this notebook claims to explain *why* Google ranks anything — only what this
  warehouse slice's own numbers say.

The cell below backs the first bullet with a number, then does the thing I said I wouldn't do —
on purpose, so the "why" for the first excluded field above isn't just asserted.

In [ ]:
# How much of the panel actually has a full Q1 feature window?
coverage = con.sql(f"""
    SELECT COUNT(*) AS n_clients_total,
           SUM(CASE WHEN gsc_data_start <= {FEATURE_START} THEN 1 ELSE 0 END) AS n_clients_with_full_q1_window
    FROM {TABLES['dim_clients']}
""").df()
print(coverage)

# --- Leakage demo: perform, then undo ---

# PERFORM: sneak the label-window outcome in as if it were a legitimate feature, then "predict"
# with it by literally re-running the label's own formula.
leaky_reconstruction = (frame['imp_label30'] < 0.8 * frame['imp_last30']).astype(int)
leak_accuracy = (leaky_reconstruction == frame['is_declining_next30']).mean()
print(f"\naccuracy reconstructing the label from the leaked column: {leak_accuracy:.3f}")
print('that is exactly 1.000 by construction -- imp_label30 IS the label\'s own numerator, so "predicting" with it is just reading the answer back.')

# UNDO: drop it, use only a feature that was knowable before 2026-04-01.
honest_rule = (frame['trend_ratio_90d'] < 0.8).astype(int)
honest_accuracy = (honest_rule == frame['is_declining_next30']).mean()
print(f'\naccuracy from a simple rule on trend_ratio_90d alone (no label-window data): {honest_accuracy:.3f}')
print(f"base rate (always predict the majority class): {max(frame['is_declining_next30'].mean(), 1 - frame['is_declining_next30'].mean()):.3f}")

honest_cols = ['imp_90d', 'pos_90d', 'trend_ratio_90d', 'days_with_impressions_90d', 'ai_referral_share_90d']
print('\ncorrelation of each pre-decision feature with the label, for comparison:')
print(frame[honest_cols + ['is_declining_next30']].corr()['is_declining_next30'].drop('is_declining_next30').sort_values(key=abs, ascending=False))


   n_clients_total  n_clients_with_full_q1_window
0              104                           40.0

accuracy reconstructing the label from the leaked column: 1.000
that is exactly 1.000 by construction -- imp_label30 IS the label's own numerator, so "predicting" with it is just reading the answer back.

accuracy from a simple rule on trend_ratio_90d alone (no label-window data): 0.469
base rate (always predict the majority class): 0.506

correlation of each pre-decision feature with the label, for comparison:
days_with_impressions_90d    0.114373
pos_90d                     -0.049706
imp_90d                     -0.037637
ai_referral_share_90d        0.024996
trend_ratio_90d              0.019310
Name: is_declining_next30, dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.